In [ ]:
!pip -q install -U groq scikit-learn pandas numpy

import getpass
import numpy as np
import pandas as pd
from groq import Groq
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("=" * 55)
print("   AI-BASED FRAUD DETECTION CHATBOT")
print("=" * 55)

# 1. Enter API key securely
api_key = getpass.getpass("Enter your Groq API key: ").strip()

if not api_key:
    raise ValueError("API key cannot be empty.")

client = Groq(api_key=api_key)

# UPDATED MODEL NAME
MODEL = "openai/gpt-oss-120b"

# 2. Create demonstration training data
np.random.seed(42)
n = 5000

X = pd.DataFrame({
    "amount": np.random.uniform(10, 100000, n),
    "hour": np.random.randint(0, 24, n),
    "location_mismatch": np.random.randint(0, 2, n),
    "international": np.random.randint(0, 2, n),
    "transactions_last_24h": np.random.randint(1, 20, n),
    "account_age_days": np.random.randint(1, 3000, n)
})

risk = (
    (X["amount"] > 50000).astype(int)
    + (X["hour"] < 5).astype(int)
    + 2 * X["location_mismatch"]
    + X["international"]
    + (X["transactions_last_24h"] > 10).astype(int)
    + (X["account_age_days"] < 30).astype(int)
)

prob = 1 / (1 + np.exp(-(risk - 3)))
y = (np.random.random(n) < prob).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("\nDemo model trained.")
print(
    "Accuracy on synthetic test data:",
    round(accuracy_score(y_test, model.predict(X_test)) * 100, 2),
    "%"
)

# 3. Store chat history
messages = [{
    "role": "system",
    "content": """
You are an AI Fraud Detection Assistant.
Explain fraud prevention in simple language.
Help users understand suspicious transactions.
Never claim a transaction is definitely fraudulent.
Model predictions are estimates, not proof.
This demonstration model uses synthetic data and is not
suitable for real-world banking decisions.
Never ask for passwords, PINs, OTPs, or full card numbers.
"""
}]

# 4. Groq-powered chatbot response
def answer_chatbot():
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages[-10:],
            temperature=0.3,
            max_completion_tokens=500
        )

        answer = response.choices[0].message.content
        print("\nAI Assistant:", answer or "No response returned.")

        if answer:
            messages.append({
                "role": "assistant",
                "content": answer
            })

    except Exception as e:
        print("\nAPI request failed:", str(e))
        print("Check your API key and model availability.")

# 5. Transaction analysis
def check_transaction():
    try:
        amount = float(input("Transaction amount (Rs.): "))
        hour = int(input("Transaction hour (0-23): "))
        location = int(input("Location mismatch? (1=yes, 0=no): "))
        international = int(input("International? (1=yes, 0=no): "))
        count = int(input("Transactions in last 24 hours: "))
        age = int(input("Account age in days: "))

        if not np.isfinite(amount) or amount <= 0:
            print("Amount must be a positive number.")
            return

        if not 0 <= hour <= 23:
            print("Hour must be between 0 and 23.")
            return

        if location not in (0, 1) or international not in (0, 1):
            print("Enter 1 for yes or 0 for no.")
            return

        if count < 1 or age < 1:
            print("Transaction count and account age must be positive.")
            return

        transaction = pd.DataFrame([{
            "amount": amount,
            "hour": hour,
            "location_mismatch": location,
            "international": international,
            "transactions_last_24h": count,
            "account_age_days": age
        }])

        prediction = int(model.predict(transaction)[0])
        probability = float(model.predict_proba(transaction)[0][1])

        result = (
            f"Model-estimated fraud risk: {probability * 100:.1f}%. "
            f"Classification: "
            f"{'Potentially suspicious' if prediction else 'Lower estimated risk'}. "
            "This is a demonstration model trained on synthetic data, "
            "not a reliable real-world banking assessment."
        )

        print("\n===== TRANSACTION ANALYSIS REPORT =====")
        print(result)

        messages.append({
            "role": "user",
            "content": "Explain this transaction result: " + result
        })

        answer_chatbot()

    except ValueError:
        print("Invalid input. Please enter valid numbers.")

# 6. Main chat loop
print("\nConnected to Groq. Type 'help' for commands.")
print("Commands: check, help, exit")

while True:
    try:
        user_text = input("\nYou: ").strip()

        if not user_text:
            continue

        command = user_text.lower()

        if command in ("exit", "quit", "bye"):
            print("Chatbot: Goodbye! Stay safe online.")
            break

        elif command == "check":
            check_transaction()

        elif command == "help":
            print("Type 'check' to analyze a sample transaction.")
            print("Ask questions about fraud prevention.")
            print("Type 'exit' to stop.")

        else:
            messages.append({
                "role": "user",
                "content": user_text
            })
            answer_chatbot()

    except (KeyboardInterrupt, EOFError):
        print("\nChatbot stopped.")
        break

   AI-BASED FRAUD DETECTION CHATBOT

Demo model trained.
Accuracy on synthetic test data: 71.3 %

Connected to Groq. Type 'help' for commands.
Commands: check, help, exit

API request failed: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
Check your API key and model availability.

API request failed: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
Check your API key and model availability.

Chatbot stopped.
